# M06 Tracing the Thoughts


## 阅读一张局部 attribution graph

加载教学 artifact，检查哪些路径对最终答案的贡献最大。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/circuits-zoom-in-fresh-20260325.git"
REPO_DIR = "circuits-zoom-in-fresh-20260325"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(candidate)], check=True)
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import json
import matplotlib.pyplot as plt

graph = json.loads((root / "artifacts" / "m06_attribution_graph.json").read_text())
case = graph["cases"][0]

fig, ax = plt.subplots(figsize=(10, 4))
for edge in case["edges"]:
    source = next(node for node in case["nodes"] if node["id"] == edge["source"])
    target = next(node for node in case["nodes"] if node["id"] == edge["target"])
    ax.plot(
        [source["x"], target["x"]],
        [source["y"], target["y"]],
        linewidth=2 + 4 * edge["score"],
        color="#c96a28",
        alpha=0.65,
    )

for node in case["nodes"]:
    color = "#123b63" if node["kind"] == "feature" else "#b5893a"
    ax.scatter(node["x"], node["y"], s=700, color=color)
    ax.text(node["x"], node["y"], node["label_en"], ha="center", va="center", color="white", fontsize=9)

ax.set_title(case["title_en"])
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
plt.tight_layout()

sorted_edges = sorted(case["edges"], key=lambda edge: edge["score"], reverse=True)
print("Top contributions:")
for edge in sorted_edges:
    print(f"  {edge['source']} -> {edge['target']}: {edge['score']:.2f}")


## 小结

tracing 的重点是找出一块忠实的计算切片，而不是把整个网络都摊出来。


## 把这本 notebook 变成研究产出

研究工作单 means this notebook is not complete when the cells finish. 请配合 /templates 里的模板，把结果写成可复查的输出。


### 运行前

- 先挑一条你准备详细解释的路径或子图。
- 先写下在这个语境里，什么叫 faithful slice of computation。
- 打开 memo 模板，预留一段“这张图还不能证明什么”。


### 运行后

- 用自己的话解释 3 条高贡献边。
- 标出一个必须靠后续实验才能消除的歧义。


### 最后交付这些产物

- 1 份 graph walkthrough。
- 1 份 ambiguity note。
- 1 个能降低这张图不确定性的下一步实验。


## 验收题

- 不用照抄图上的标签，解释这张 attribution graph 里最重要的 3 条贡献路径在做什么。
- 这张图支持什么结论，又明确不支持什么结论？
- 如果你只能再做一个后续实验来降低这张图的歧义，你会选什么？
- 如果你不能在不重看讲义的情况下独立答出其中至少 2 题，就回去重看文章和你的书面输出。
